<a href="https://colab.research.google.com/github/dawamukherjee/delhi-aqi-predictor/blob/main/03_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AirWise AI — Minimal White Dashboard

Google Colab demonstration. This notebook includes the full frontend and machine-learning pipeline.

**Important:** PM2.5 concentration is measured in µg/m³; it is not the same as the AQI index. The chart supports readings above 700 µg/m³, but the model does not invent high forecasts. It forecasts historical data six hours ahead.


In [ ]:
!pip -q install "pandas>=2.1,<3" "numpy>=1.26,<3" "scikit-learn>=1.4,<2" "gradio>=5,<7" "plotly>=5.20,<7" "requests>=2.31,<3"

## Dashboard implementation
The following cell contains the **entire app**. Change the design, model, and copy here; no external `.py` uploads are necessary.

In [ ]:
%%writefile airwise_app.py
"""AirWise AI — self-contained historical PM2.5 forecasting dashboard.
Sources: Open-Meteo air-quality and weather historical endpoints.
A prototype, NOT a real-time or indoor safety controller.
"""
import datetime as dt
import functools
import html
import os
import warnings
import requests
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import gradio as gr
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
warnings.filterwarnings('ignore', category=FutureWarning)

LAT, LON = 28.6139, 77.2090
TZ = 'Asia/Kolkata'
HORIZON = 6

CSS = """
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700;800&display=swap');
:root{--surface:#fff;--page:#f7f8fa;--ink:#202b3a;--muted:#697889;--line:#e7eaf0;--blue:#315b89}
body,.gradio-container,.main{background:#f7f8fa!important;color:#202b3a!important}
.gradio-container{font-family:Inter,Arial,sans-serif!important;max-width:1400px!important;margin:auto!important}
#shell{padding:4px 15px 38px}#header{margin-bottom:14px}#header .prose{max-width:none}
.logo-row{display:flex;align-items:center;justify-content:space-between;gap:12px;flex-wrap:wrap;border-bottom:1px solid #e5e8ec;padding:17px 3px 19px}
.brand{display:flex;align-items:center;gap:11px;font-size:22px;font-weight:800;letter-spacing:-.9px;color:#202b3a}
.brand-icon{display:grid;place-items:center;width:36px;height:36px;border:1px solid #dae2eb;border-radius:10px;background:#eef3fa;color:#315b89;font-size:20px}.brand span{color:#315b89}
.chip{border:1px solid #e0e5ed;background:#fff;border-radius:100px;padding:8px 13px;color:#637388;font-size:11px;font-weight:600;letter-spacing:.01em}
.hero{padding:14px 3px 24px}.eyebrow{font-size:11px;text-transform:uppercase;letter-spacing:1.6px;font-weight:700;color:#587394}
.hero h1{font-size:clamp(28px,4vw,41px);font-weight:750;letter-spacing:-1.9px;line-height:1.15;color:#202b3a;margin:8px 0 10px}.hero p{color:#6d7886;font-size:13px;max-width:710px;line-height:1.75}
#config,#forecastPanel,#methodsPanel,#chartPanel,#comparisonPanel{background:#fff!important;border:1px solid #e6e9ef!important;border-radius:15px!important;padding:20px!important;box-shadow:0 3px 12px #14253b08!important}
.section-heading{font-size:15px;font-weight:750;letter-spacing:-.25px;color:#273344;margin:0 0 6px}.section-sub{font-size:12px;color:#84909d;line-height:1.6;margin-bottom:15px}.small-label{font-size:12px;color:#6e7e8f}
#config label span,#config label,#config .wrap{color:#455366!important}#config input{color:#253349!important;background:#fff!important}
#action{background:#315b89!important;color:#fff!important;font-weight:700!important;border:0!important;border-radius:9px!important;min-height:45px;box-shadow:none!important}
#action:hover{background:#264a72!important}
.metrics{display:grid;grid-template-columns:repeat(4,minmax(0,1fr));gap:10px;margin-bottom:14px}.metric{border:1px solid #e5e9ef;border-radius:12px;background:#fff;padding:16px}.metric .label{font-size:11px;color:#748191;font-weight:500}.metric .value{font-size:25px;font-weight:750;color:#243144;letter-spacing:-1px;margin:10px 0 4px}.metric .unit{font-size:11px;font-weight:500;color:#8190a0;letter-spacing:0}.metric .note{font-size:10px;color:#8b98a6}.metric.accent{background:#f0f5fa;border-color:#dce6f1}.metric.accent .value{color:#315b89}
.verdict{padding:20px;border:1px solid #dde7f0;border-radius:12px;background:#f4f8fc;margin-bottom:17px;display:flex;align-items:center;justify-content:space-between;gap:20px;flex-wrap:wrap}.verdict.off{background:#f8f9fb;border-color:#e4e7ee}.verdict .status{font-size:20px;font-weight:750;color:#254567}.verdict .desc{color:#65758a;font-size:12px;max-width:650px;line-height:1.7;margin-top:7px}.verdict .symbol{font-size:31px;color:#6783a3}
.report-grid{display:grid;grid-template-columns:repeat(3,1fr);gap:11px}.mini{padding:16px;background:#fafbfd;border:1px solid #e7ebf0;border-radius:11px}.mini b{display:block;color:#27384b;font-size:19px;margin:8px 0}.mini span{color:#7a8798;font-size:11px}.method-list{color:#6a7788;font-size:12px;line-height:2}.method-list strong{color:#27394d}.smallprint{font-size:11px;color:#8995a3;line-height:1.7;margin-top:12px}.error{color:#9b3434;border:1px solid #ebc6c6;background:#fff7f7;padding:18px;border-radius:12px}
#forecastChart{background:#fff!important}#forecastChart .plot-container{border-radius:12px;overflow:hidden}.gradio-container button:not(#action){border-radius:10px!important}
@media(max-width:900px){.metrics{grid-template-columns:repeat(2,minmax(0,1fr))}.report-grid{grid-template-columns:1fr}.metric .value{font-size:23px}}
@media(max-width:470px){.metrics{grid-template-columns:repeat(2,minmax(0,1fr));gap:8px}.metric{padding:12px}.metric .value{font-size:20px}}
"""

def get_json(url, params):
    response = requests.get(url, params=params, timeout=35, headers={'User-Agent':'AirWiseAI-Hackathon/1.0'})
    response.raise_for_status()
    payload=response.json()
    if 'hourly' not in payload:
        raise RuntimeError(payload.get('reason', 'No hourly data returned'))
    return payload['hourly']

def load_data():
    # Explicit historical dates make the demonstration reproducible and do not imply live data.
    end = dt.date.today() - dt.timedelta(days=7)
    start = end - dt.timedelta(days=100)
    dates={'latitude':LAT,'longitude':LON,'start_date':str(start),'end_date':str(end),'timezone':TZ}
    air = get_json('https://air-quality-api.open-meteo.com/v1/air-quality', {
        **dates,'hourly':'pm2_5,pm10'})
    weather = get_json('https://archive-api.open-meteo.com/v1/archive', {
        **dates,'hourly':'temperature_2m,relative_humidity_2m,wind_speed_10m,precipitation'})
    a=pd.DataFrame(air).rename(columns={'pm2_5':'pm25'})
    w=pd.DataFrame(weather)
    a['time']=pd.to_datetime(a['time']); w['time']=pd.to_datetime(w['time'])
    d=a.merge(w,on='time',how='inner').sort_values('time').reset_index(drop=True)
    for col in ['pm25','pm10','temperature_2m','relative_humidity_2m','wind_speed_10m','precipitation']:
        d[col]=pd.to_numeric(d[col],errors='coerce')
    d=d.dropna(subset=['pm25']).copy()
    d=d.set_index('time').asfreq('h')
    # Interpolate short gaps only; do not make large gaps look like measured data.
    d[['pm25','pm10','temperature_2m','relative_humidity_2m','wind_speed_10m','precipitation']]=d[['pm25','pm10','temperature_2m','relative_humidity_2m','wind_speed_10m','precipitation']].interpolate(limit=2)
    d=d.reset_index()
    if len(d)<250:raise RuntimeError('Insufficient hourly historical data for evaluation')
    return d

FEATURES=['pm25','pm10','temperature_2m','relative_humidity_2m','wind_speed_10m','precipitation','hour_sin','hour_cos','day_sin','day_cos','pm_lag_1','pm_lag_3','pm_lag_6','pm_roll_6']

def make_features(df):
    d=df.copy()
    h=d.time.dt.hour
    d['hour_sin']=np.sin(2*np.pi*h/24);d['hour_cos']=np.cos(2*np.pi*h/24)
    day=d.time.dt.dayofyear
    d['day_sin']=np.sin(2*np.pi*day/365.25);d['day_cos']=np.cos(2*np.pi*day/365.25)
    for n in [1,3,6]:d[f'pm_lag_{n}']=d.pm25.shift(n)
    d['pm_roll_6']=d.pm25.shift(1).rolling(6).mean()
    d['target']=d.pm25.shift(-HORIZON)
    return d

@functools.lru_cache(maxsize=1)
def trained():
    raw=load_data()
    featured=make_features(raw)
    all_rows=featured.dropna(subset=FEATURES+['target']).copy()
    # Chronological 70% train, 10% validation, 20% untouched test.
    n=len(all_rows);tr=int(n*.70);va=int(n*.80)
    train=all_rows.iloc[:tr-HORIZON]; validation=all_rows.iloc[tr:va-HORIZON]; test=all_rows.iloc[va:]
    # Hyperparameters are fixed; validation is retained for inspection and not used in tuning.
    model=RandomForestRegressor(n_estimators=180,max_depth=14,min_samples_leaf=3,random_state=42,n_jobs=-1)
    model.fit(train[FEATURES],train['target'])
    test=test.copy();test['prediction']=np.maximum(0,model.predict(test[FEATURES]))
    test['baseline']=test['pm25']
    model_mae=mean_absolute_error(test.target,test.prediction)
    baseline_mae=mean_absolute_error(test.target,test.baseline)
    latest=featured.dropna(subset=FEATURES).iloc[-1]
    prediction=max(0,float(model.predict(pd.DataFrame([latest[FEATURES]])[FEATURES])[0]))
    return {'raw':raw,'test':test,'latest':latest,'prediction':prediction,'model_mae':model_mae,'baseline_mae':baseline_mae,'validation_count':len(validation),'train_count':len(train),'test_count':len(test)}

def esc(x):return html.escape(str(x))

def metric(label,value,note='',accent=False):
    return f'<div class="metric {"accent" if accent else ""}"><div class="label">{label}</div><div class="value">{value}</div><div class="note">{note}</div></div>'

def chart(result,trigger):
    d=result['raw'].dropna(subset=['pm25']).tail(60)
    last=result['latest'];forecast_time=last.time+pd.Timedelta(hours=HORIZON)
    fig=go.Figure()
    fig.add_trace(go.Scatter(x=d.time,y=d.pm25,mode='lines',name='Observed PM2.5',line=dict(color='#547da7',width=3),fill='tozeroy',fillcolor='rgba(84,125,167,.07)',hovertemplate='%{x|%d %b %H:%M}<br>%{y:.1f} µg/m³<extra></extra>'))
    fig.add_trace(go.Scatter(x=[last.time,forecast_time],y=[last.pm25,result['prediction']],mode='lines+markers',name='AI six-hour forecast',line=dict(color='#bb6b52',width=3,dash='dash'),marker=dict(size=[8,13]),hovertemplate='%{x|%d %b %H:%M}<br>%{y:.1f} µg/m³<extra></extra>'))
    fig.add_hline(y=float(trigger),line_dash='dot',line_color='#9da9b6',annotation_text='Decision threshold',annotation_font_color='#68798a')
    fig.update_layout(height=355,margin=dict(l=22,r=22,t=18,b=20),paper_bgcolor='#ffffff',plot_bgcolor='#ffffff',font=dict(color='#68798a',family='Inter'),xaxis=dict(showgrid=False,zeroline=False,title=None),yaxis=dict(gridcolor='#e9edf2',zeroline=False,title='PM2.5 concentration (µg/m³)',range=[0,max(800, float(d.pm25.max())*1.12, float(result['prediction'])*1.12, float(trigger)*1.12)]),legend=dict(orientation='h',y=1.18,x=0),hovermode='x unified')
    return fig

def render(power,tariff,trigger):
    try:
        power=float(power);tariff=float(tariff);trigger=float(trigger)
        if not(5<=power<=2000 and 0<=tariff<=100 and 0<=trigger<=1000):raise ValueError('Please provide valid power, tariff and trigger values.')
        r=trained();test=r['test'];latest=r['latest'];pred=r['prediction'];on=pred>=trigger
        actual_bad=test.target>=trigger
        ai_on=test.prediction>=trigger
        rule_on=test.pm25>=trigger
        hours=len(test);month_hours=720
        ai_fraction=float(ai_on.mean());base_fraction=float(rule_on.mean())
        always_kwh=power*month_hours/1000
        scheduled_kwh=always_kwh*ai_fraction
        saved_kwh=always_kwh-scheduled_kwh
        saved_rupees=saved_kwh*tariff
        missed=int((actual_bad & ~ai_on).sum());covered=int((actual_bad & ai_on).sum()); total_bad=int(actual_bad.sum())
        baseline_missed=int((actual_bad & ~rule_on).sum())
        ts=pd.Timestamp(latest.time)
        target_time=ts+pd.Timedelta(hours=HORIZON)
        cards='<div class="metrics">'+''.join([
            metric('Observed PM2.5',f'{latest.pm25:.1f}<span class="unit"> µg/m³</span>',f'Observed · {ts:%d %b %Y, %H:%M}'),
            metric('Predicted PM2.5 · 6h',f'{pred:.1f}<span class="unit"> µg/m³</span>',f'Target · {target_time:%d %b, %H:%M}',True),
            metric('Estimated monthly saving',f'₹{saved_rupees:,.0f}',f'{saved_kwh:.1f} kWh · simulated'),
            metric('Forecast error · MAE',f'{r["model_mae"]:.1f}<span class="unit"> µg/m³</span>',f'Baseline: {r["baseline_mae"]:.1f} µg/m³')
        ])+'</div>'
        title='KEEP PURIFIER ON' if on else 'BELOW SELECTED TRIGGER'
        desc=(f'Predicted outdoor PM2.5 ({pred:.1f} µg/m³) is above the selected {trigger:.0f} µg/m³ threshold.' if on else f'Predicted outdoor PM2.5 ({pred:.1f} µg/m³) is below the selected {trigger:.0f} µg/m³ threshold. Do not automatically turn a real purifier off based on outdoor air alone.')
        verdict=f'<div class="verdict {"" if on else "off"}"><div><div class="eyebrow">Six-hour recommendation · simulation only</div><div class="status">{title}</div><div class="desc">{esc(desc)}<br>Indoor air quality is not measured; this is not a safety instruction.</div></div><div class="symbol">{"↑" if on else "—"}</div></div>'
        energy='<div class="report-grid">'+''.join([
            f'<div class="mini"><span>Always running · 30 days</span><b>{always_kwh:.1f} kWh</b><span>₹{always_kwh*tariff:,.0f} estimated cost</span></div>',
            f'<div class="mini"><span>AI scheduled · 30 days</span><b>{scheduled_kwh:.1f} kWh</b><span>₹{scheduled_kwh*tariff:,.0f} estimated cost</span></div>',
            f'<div class="mini"><span>Estimated runtime reduction</span><b>{(1-ai_fraction)*100:.1f}%</b><span>Equivalent monthly saving: ₹{saved_rupees:,.0f}</span></div>'
        ])+'</div>'
        performance='<div class="report-grid">'+''.join([
            f'<div class="mini"><span>Model error (MAE)</span><b>{r["model_mae"]:.1f} µg/m³</b><span>Random Forest · held-out test</span></div>',
            f'<div class="mini"><span>Persistence baseline error</span><b>{r["baseline_mae"]:.1f} µg/m³</b><span>Assumes PM2.5 stays unchanged</span></div>',
            f'<div class="mini"><span>Hours above trigger covered</span><b>{covered} / {total_bad}</b><span>{missed} missed · chosen trigger</span></div>',
        ])+'</div>'+f'<p class="smallprint">Test size: {r["test_count"]} hourly predictions. Current-reading rule missed {baseline_missed} high-pollution target hours; AI forecast rule missed {missed}. Estimated energy saved by current-reading rule: {(1-base_fraction)*100:.1f}% versus {100*(1-ai_fraction):.1f}% by the AI rule. These are historical outdoor-air simulations, not observed indoor exposure or measured electricity savings.</p>'
        methodology=f'<div class="method-list"><strong>Model:</strong> Random Forest Regressor · 180 trees · 6-hour forecast horizon<br><strong>Inputs:</strong> PM2.5, PM10, temperature, humidity, wind, precipitation, time cycles, lagged PM2.5<br><strong>Source:</strong> Open-Meteo archived Delhi air-quality and weather data<br><strong>Evaluation:</strong> chronological 70% train / 10% validation / 20% untouched test; train {r["train_count"]:,} rows, validation {r["validation_count"]:,} rows, test {r["test_count"]:,} rows<br><strong>Policy:</strong> predicted outdoor PM2.5 ≥ chosen threshold means a simulated purifier ON decision; comparison uses actual six-hour-ahead outdoor PM2.5<br><strong>Data timestamp:</strong> latest archived observation {ts:%d %B %Y, %H:%M} (Asia/Kolkata)</div><p class="smallprint">Research prototype. This interface retrieves historical data when the Colab session starts, not a live PM2.5 measurement. Changing the threshold is exploratory and does not constitute independently validated control-policy optimization. No indoor PM2.5 measurements, device control, standby energy, or variable fan speeds are included. Avoid switching off devices solely on this prediction.</p>'
        return cards,verdict,chart(r,trigger),energy,performance,methodology
    except Exception as exc:
        msg=f'<div class="error"><strong>Unable to calculate the forecast</strong><br>{esc(exc)}<br>Check Colab internet access and try again. This demo requires Open-Meteo historical API access.</div>'
        return msg,'',go.Figure(),'','',''

with gr.Blocks(title='AirWise AI · Delhi Pollution Forecast',css=CSS,theme=gr.themes.Base(primary_hue='blue',neutral_hue='slate')) as demo:
    with gr.Column(elem_id='shell'):
        gr.HTML('''<div id="header"><div class="logo-row"><div class="brand"><div class="brand-icon">◌</div> AirWise <span>AI</span></div><div class="chip">AI + CLIMATE  /  RESEARCH PROTOTYPE</div></div></div>''')
        gr.HTML('''<div class="hero"><div class="eyebrow">DELHI  /  AIR QUALITY FORECAST</div><h1>Understand pollution.<br><span style="color:#315b89">Plan energy use.</span></h1><p>Explore a machine-learning forecast of outdoor particulate pollution and simulate when an air purifier could operate. A historical-data research prototype for the Veritas AI 2026 hackathon.</p></div>''')
        with gr.Row(equal_height=False):
            with gr.Column(scale=1,min_width=280,elem_id='config'):
                gr.HTML('<div class="section-heading">Your purifier setup</div><div class="section-sub">Adjust assumptions to explore forecast-based energy use.</div>')
                power=gr.Slider(20,200,value=60,step=5,label='Purifier power (W)')
                tariff=gr.Number(value=8,precision=2,label='Electricity price (₹/kWh)')
                trigger=gr.Slider(10,1000,value=60,step=10,label='PM2.5 decision threshold (µg/m³)')
                run=gr.Button('Generate forecast',variant='primary',elem_id='action')
                gr.HTML('<div class="smallprint">Forecasts are evaluated against stored historical observations, not a live sensor. The threshold is a configurable experiment, not a medical or safety standard.</div>')
            with gr.Column(scale=3,min_width=380,elem_id='forecastPanel'):
                cards=gr.HTML()
                recommendation=gr.HTML()
                gr.HTML('<div class="section-heading">PM2.5 concentration</div><div class="section-sub">Last 60 observed hours and one six-hour-ahead model prediction. Scale accommodates extreme readings over 700 µg/m³; these values are concentrations, not AQI index points.</div>')
                figure=gr.Plot(elem_id='forecastChart',show_label=False)
        with gr.Tabs():
            with gr.Tab('Energy simulation'):
                with gr.Column(elem_id='comparisonPanel'):
                    gr.HTML('<div class="section-heading">What could smarter scheduling save?</div><div class="section-sub">Extrapolated from the forecast-rule operating fraction on the test period.</div>')
                    energy_html=gr.HTML()
            with gr.Tab('Model evaluation'):
                with gr.Column(elem_id='comparisonPanel'):
                    gr.HTML('<div class="section-heading">Measured against a simple baseline</div><div class="section-sub">A separate, chronologically held-out test period provides a fairer comparison.</div>')
                    performance_html=gr.HTML()
            with gr.Tab('Data and methods'):
                with gr.Column(elem_id='methodsPanel'):
                    gr.HTML('<div class="section-heading">How this prototype works</div>')
                    method_html=gr.HTML()
        gr.HTML('<p class="smallprint" style="padding:12px 4px">AirWise AI · Historical-data research prototype · Weather/air quality: Open-Meteo · Powered by scikit-learn, Plotly, Gradio.</p>')
    outputs=[cards,recommendation,figure,energy_html,performance_html,method_html]
    run.click(fn=render,inputs=[power,tariff,trigger],outputs=outputs)
    demo.load(fn=render,inputs=[power,tariff,trigger],outputs=outputs)

if __name__=='__main__':
    demo.launch(share=os.getenv('GRADIO_SHARE','false').lower()=='true',show_error=True)


## Launch the dashboard
Run the next cell to open a temporary Gradio link. This is a historical-data forecast, not a live sensor reading.


In [ ]:
import os
os.environ['GRADIO_SHARE'] = 'true'
%run airwise_app.py